# CALCULOS DE PM10 PM2.5 CO

Calcula, a partir de `CEN_marzo.csv`:
- **PM10 NowCast** (promedio móvil ponderado de 12 h)
- **PM2.5 NowCast** (promedio móvil ponderado de 12 h)
- **CO**: promedio móvil de 8 h
- **Categorías horarias** del Índice AIRE Y SALUD para O3, CO, PM10, PM2.5, NO2, SO2
- **Categoría global** (la más desfavorable de las anteriores)

Todo conforme a NOM-172-SEMARNAT-2023. Los pasos intermedios del NowCast (rango, factor de ponderación, suma ponderada) se calculan dentro de una función de Python — no se muestran como columnas, para que la tabla final quede limpia.

## 1. Subida de archivo

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('CEN_marzo.csv')
df.columns = [c.strip() for c in df.columns]
df['DATE'] = pd.to_datetime(df['DATE'])
df = df.sort_values(['DATE']).reset_index(drop=True)
df.head()

,STATION,DATE,HOUR,DIA,MES,O3,NO,NO2,NOX,SO2,...,IT,ET,RH,WS,WD,PP,ATM,RS,UVI,promedio móvil simple de 8 horas CO
0,CEN,2024-03-01 00:00:00,0,1,3,0.025,NaN,NaN,NaN,NaN,...,23.79,18.175,NaN,NaN,260.1,NaN,667.59,NaN,NaN,Requiere 8 Horas de dato
1,CEN,2024-03-01 01:00:00,1,1,3,0.024,NaN,NaN,NaN,NaN,...,23.71,17.359,NaN,NaN,282.6,NaN,665.17,NaN,NaN,Requiere 8 Horas de dato
2,CEN,2024-03-01 02:00:00,2,1,3,0.021,NaN,NaN,NaN,NaN,...,23.68,16.622,NaN,NaN,345.1,NaN,662.91,NaN,NaN,Requiere 8 Horas de dato
3,CEN,2024-03-01 03:00:00,3,1,3,0.023,NaN,NaN,NaN,NaN,...,23.69,15.659,NaN,NaN,327.2,NaN,661.32,NaN,NaN,Requiere 8 Horas de dato
4,CEN,2024-03-01 04:00:00,4,1,3,0.017,NaN,NaN,NaN,NaN,...,23.70,14.186,NaN,NaN,319.2,NaN,661.00,NaN,NaN,Requiere 8 Horas de dato


## 2.Calculo de (NowCast, promedio 8h y categorías NOM-172)

In [ ]:
def nowcast_una_ventana(valores):
    valores = pd.Series(valores, dtype='float64')

    # Requisito de compleción: al menos 2 de las 3 horas mas recientes
    if valores.iloc[-3:].notna().sum() < 2:
        return np.nan

    validos = valores.dropna()
    if validos.empty:
        return np.nan

    cmax, cmin = validos.max(), validos.min()
    rango = cmax - cmin

    if cmax == 0:
        w = 1.0
    else:
        w = max(0.5, round(1 - rango / cmax, 2))

    n = len(valores)
    suma_pond, suma_pesos = 0.0, 0.0
    for i, val in enumerate(valores):
        if pd.isna(val):
            continue
        horas_atras = (n - 1) - i
        peso = w ** horas_atras
        suma_pond += val * peso
        suma_pesos += peso

    if suma_pesos == 0:
        return np.nan
    return round(suma_pond / suma_pesos)


def calcular_nowcast(serie, ventana=12):
    """Aplica nowcast_una_ventana de forma deslizante sobre toda la serie horaria."""
    valores = serie.values
    resultado = [np.nan] * len(valores)
    for i in range(len(valores)):
        if i < ventana - 1:
            continue
        resultado[i] = nowcast_una_ventana(valores[i - ventana + 1: i + 1])
    return pd.Series(resultado, index=serie.index)


def promedio_movil_co(serie, ventana=8, minimo=6):
    """Promedio movil simple de 8 horas para CO, requiere al menos 6 de 8 horas validas."""
    return serie.rolling(window=ventana, min_periods=minimo).mean().round(2)


# --- Tablas de categorias (NOM-172-SEMARNAT-2023, vigentes desde enero 2024) ---

def cat_o3(x):
    if pd.isna(x): return 'NA'
    if x < 0.058: return 'Buena'
    if x <= 0.090: return 'Aceptable'
    if x <= 0.135: return 'Mala'
    if x <= 0.175: return 'Muy Mala'
    return 'Extremadamente Mala'

def cat_co(x):
    if pd.isna(x): return 'NA'
    if x < 5: return 'Buena'
    if x <= 9: return 'Aceptable'
    if x <= 12: return 'Mala'
    if x <= 16: return 'Muy Mala'
    return 'Extremadamente Mala'

def cat_pm10(x):
    if pd.isna(x): return 'NA'
    if x < 45: return 'Buena'
    if x <= 60: return 'Aceptable'
    if x <= 132: return 'Mala'
    if x <= 213: return 'Muy Mala'
    return 'Extremadamente Mala'

def cat_pm25(x):
    if pd.isna(x): return 'NA'
    if x < 15: return 'Buena'
    if x <= 33: return 'Aceptable'
    if x <= 79: return 'Mala'
    if x <= 130: return 'Muy Mala'
    return 'Extremadamente Mala'

def cat_no2(x):
    if pd.isna(x): return 'NA'
    if x < 0.053: return 'Buena'
    if x <= 0.106: return 'Aceptable'
    if x <= 0.160: return 'Mala'
    if x <= 0.213: return 'Muy Mala'
    return 'Extremadamente Mala'

def cat_so2(x):
    if pd.isna(x): return 'NA'
    if x < 0.035: return 'Buena'
    if x <= 0.075: return 'Aceptable'
    if x <= 0.185: return 'Mala'
    if x <= 0.304: return 'Muy Mala'
    return 'Extremadamente Mala'

RANGO_BANDAS = {'Buena':1, 'Aceptable':2, 'Mala':3, 'Muy Mala':4, 'Extremadamente Mala':5}

def categoria_global(row):
    cats = [row['Categoria_O3'], row['Categoria_CO'], row['Categoria_PM10'], row['Categoria_PM2.5'],
            row['Categoria_NO2'], row['Categoria_SO2']]
    cats = [c for c in cats if c != 'NA']
    if not cats:
        return 'NA'
    return max(cats, key=lambda c: RANGO_BANDAS[c])

## 3. Aplicar los calculos

In [ ]:
resultado = df[['STATION', 'DATE', 'HOUR']].copy()

# Concentraciones base horarias
resultado['O3'] = df['O3']
resultado['CO_prom_8h'] = promedio_movil_co(df['CO'])
resultado['PM10_NowCast'] = calcular_nowcast(df['PM10'])
resultado['PM2.5_NowCast'] = calcular_nowcast(df['PM2.5'])
resultado['NO2'] = df['NO2'] if 'NO2' in df.columns else np.nan
resultado['SO2'] = df['SO2'] if 'SO2' in df.columns else np.nan

# Categorias horarias por contaminante
resultado['Categoria_O3'] = resultado['O3'].apply(cat_o3)
resultado['Categoria_CO'] = resultado['CO_prom_8h'].apply(cat_co)
resultado['Categoria_PM10'] = resultado['PM10_NowCast'].apply(cat_pm10)
resultado['Categoria_PM2.5'] = resultado['PM2.5_NowCast'].apply(cat_pm25)
resultado['Categoria_NO2'] = resultado['NO2'].apply(cat_no2)
resultado['Categoria_SO2'] = resultado['SO2'].apply(cat_so2)

# Categoria global de la hora = la mas desfavorable
resultado['Categoria_Global'] = resultado.apply(categoria_global, axis=1)

resultado.head(20)

,STATION,DATE,HOUR,O3,CO_prom_8h,PM10_NowCast,PM2.5_NowCast,NO2,SO2,Categoria_O3,Categoria_CO,Categoria_PM10,Categoria_PM2.5,Categoria_NO2,Categoria_SO2,Categoria_Global
0,CEN,2024-03-01 00:00:00,0,0.025,NaN,NaN,NaN,NaN,NaN,Buena,NA,NA,NA,NA,NA,Buena
1,CEN,2024-03-01 01:00:00,1,0.024,NaN,NaN,NaN,NaN,NaN,Buena,NA,NA,NA,NA,NA,Buena
2,CEN,2024-03-01 02:00:00,2,0.021,NaN,NaN,NaN,NaN,NaN,Buena,NA,NA,NA,NA,NA,Buena
3,CEN,2024-03-01 03:00:00,3,0.023,NaN,NaN,NaN,NaN,NaN,Buena,NA,NA,NA,NA,NA,Buena
4,CEN,2024-03-01 04:00:00,4,0.017,NaN,NaN,NaN,NaN,NaN,Buena,NA,NA,NA,NA,NA,Buena
5,CEN,2024-03-01 05:00:00,5,0.002,0.97,NaN,NaN,NaN,NaN,Buena,Buena,NA,NA,NA,NA,Buena
6,CEN,2024-03-01 06:00:00,6,0.001,1.15,NaN,NaN,NaN,NaN,Buena,Buena,NA,NA,NA,NA,Buena
7,CEN,2024-03-01 07:00:00,7,0.001,1.40,NaN,NaN,NaN,NaN,Buena,Buena,NA,NA,NA,NA,Buena
8,CEN,2024-03-01 08:00:00,8,0.010,1.58,NaN,NaN,NaN,NaN,Buena,Buena,NA,NA,NA,NA,Buena
9,CEN,2024-03-01 09:00:00,9,0.027,1.69,NaN,NaN,NaN,NaN,Buena,Buena,NA,NA,NA,NA,Buena


## 4. Revisión rápida

In [ ]:
print('Filas totales:', len(resultado))
print()
print('Distribución Categoría Global:')
print(resultado['Categoria_Global'].value_counts())

Filas totales: 744

Distribución Categoría Global:
Categoria_Global
Buena        453
Aceptable    168
Mala         123
Name: count, dtype: int64


## 5. Guardar y descargar resultado

In [ ]:
resultado.to_csv('CEN_marzo_Indice_AIRE_Y_SALUD.csv', index=False)
files.download('CEN_marzo_Indice_AIRE_Y_SALUD.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>